# 02 · GSM8k evaluation (1319 test questions, greedy, 512 new tokens)
Same prompt and answer extraction as the original `evaluate_gsm8k.py`. We report accuracy on all 1319 questions, and also on the
first 1312 to match the original script, which drops the last partial batch.

Two backends:
* **vLLM** (fast): merges the adapter into the weights first (`merge.py`). It needs a *fresh runtime*, because installing vLLM replaces torch.
* **HF generate** (fallback, about 20–40 min per run): uses the adapter directly and is resumable.

Run the **parity check** once, so that numbers from the two backends can be compared.

In [ ]:
# ── Common setup (run every session) ─────────────────────────────────────────
REPO_URL = "https://github.com/Mrazakos/DiaBlo.git"   # fork with the colab-repro branch
BRANCH   = "colab-repro"
MODEL    = "Qwen2.5-1.5B"                                       # short name or HF id
HF_ID    = "Qwen/Qwen2.5-1.5B"

import os, subprocess, torch
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/diablo"
os.makedirs(DRIVE, exist_ok=True)
DATA    = f"{DRIVE}/datasets/{MODEL}"     # tokenized data is tokenizer-specific
RESULTS = f"{DRIVE}/results"
EVALS   = f"{DRIVE}/eval"
os.environ["HF_HOME"] = f"{DRIVE}/hf_cache"   # keep downloads across sessions
os.environ["WANDB_MODE"] = "disabled"

REPO = "/content/DiaBlo"
if not os.path.exists(REPO):
    if os.path.exists(f"{DRIVE}/DiaBlo"):          # alternative: repo uploaded to Drive
        subprocess.run(["cp", "-r", f"{DRIVE}/DiaBlo", REPO], check=True)
    else:
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO], check=True)
%cd {REPO}
!git log -1 --oneline 2>/dev/null || true
# Colab ships torchao 0.10; recent peft refuses to build LoRA layers when an old torchao is
# installed (we don't use torchao), so remove it.
!pip -q uninstall -y torchao
!pip -q install peft
print(torch.__version__, torch.cuda.get_device_name(0), "bf16:", torch.cuda.is_bf16_supported())

In [ ]:
BACKEND = "vllm"   # "vllm" or "hf"
RUN_IDS = ["R0", "R1", "R2", "R3", "R4"]   # R0 = base model, no fine-tuning
if BACKEND == "vllm":
    !pip -q install vllm

In [ ]:
import json, os, subprocess
os.makedirs(EVALS, exist_ok=True)

def eval_run(run_id, backend=BACKEND, limit=None, tag=""):
    out = f"{EVALS}/{run_id}{tag}_{backend}.jsonl"
    lim = f"--limit {limit}" if limit else ""
    if run_id == "R0":
        if backend == "vllm":
            cmd = f"python eval_gsm8k_vllm.py --model_path {HF_ID} --data_dir {DATA} --output_file {out} {lim}"
        else:
            cmd = f"python evaluate_gsm8k.py --model_name {MODEL} --method none --data_dir {DATA} --precision fp16 --batch_size 32 --output_file {out} {lim}"
    else:
        run_dir = f"{RESULTS}/{run_id}"
        cfg = json.load(open(f"{run_dir}/run_config.json"))
        assert os.path.exists(f"{run_dir}/adapter.chkpt"), f"{run_id} not finished"
        if backend == "vllm":
            merged = f"/content/merged/{run_id}"            # local disk, not Drive (~3GB)
            if not os.path.exists(merged):
                subprocess.run(f"python merge.py --run_dir {run_dir} --output_dir {merged} --precision fp16", shell=True, check=True)
            cmd = f"python eval_gsm8k_vllm.py --model_path {merged} --data_dir {DATA} --output_file {out} {lim}"
        else:
            m = cfg["method"]
            size = f"--num_blocks {cfg['num_blocks']}" if m == "diablo" else f"--lora_r {cfg['lora_r']}"
            cmd = (f"python evaluate_gsm8k.py --model_name {MODEL} --method {m} {size} --target_modules {cfg['target_modules']} "
                   f"--adapter_path {run_dir}/adapter.chkpt --data_dir {DATA} --precision fp16 --batch_size 32 --output_file {out} {lim}")
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)
    return json.load(open(out.replace(".jsonl", "") + "_summary.json"))

### Parity check (once): vLLM vs HF on the first 64 questions of R1

In [ ]:
# Run with BACKEND="vllm" installed. The HF backend also works in the same runtime.
s_v = eval_run("R1", "vllm", limit=64, tag="_parity64")
s_h = eval_run("R1", "hf",   limit=64, tag="_parity64")
a = [json.loads(l) for l in open(f"{EVALS}/R1_parity64_vllm.jsonl")]
b = {r["idx"]: r for r in map(json.loads, open(f"{EVALS}/R1_parity64_hf.jsonl"))}
agree = sum(r["correct"] == b[r["idx"]]["correct"] for r in a)
print(f"vLLM acc {s_v['accuracy']:.3f} | HF acc {s_h['accuracy']:.3f} | per-question agreement {agree}/64")
# fp16 kernels differ, so expect a few disagreements. A large gap means something is wrong.

### Full evaluation

In [ ]:
summaries = {}
for rid in RUN_IDS:
    out = f"{EVALS}/{rid}_{BACKEND}_summary.json"
    if os.path.exists(out):
        summaries[rid] = json.load(open(out)); continue
    try:
        summaries[rid] = eval_run(rid)
    except Exception as e:
        print(rid, "skipped:", e)
for rid, s in summaries.items():
    print(f"{rid}: acc {s['accuracy']*100:.2f}% (n={s['n']}) | orig-protocol {s['accuracy_orig_protocol']*100:.2f}%")